#### Model Training

In [ ]:
from ultralytics import YOLO
from yolo_train import clear_training_output

# Load a pretrained YOLO model (recommended for training)
model = YOLO("yolov8n.pt")

# Delete the previous weights and logs only when starting this training cell.
# New results always use runs/train; no backup is kept.
train_dir = clear_training_output()

# Train the model on a custom dataset
# 'data' points to the dataset configuration file
# 'batch=32' uses a fixed training batch size
# 'project' and 'name' specify where to save training results
# In notebooks, workers=0 avoids starting data-loading subprocesses.
# For parallel loading, run yolo_train.py --workers 4 from a terminal.
model.train(
    data="dataset/data.yaml",
    batch=32,
    workers=0,
    project=str(train_dir.parent),  # Absolute output root
    name=train_dir.name,  # Always runs/train
    exist_ok=True
)

#### Video Inference and Frame Counting

In [ ]:
from ultralytics import YOLO

# Load the best weights from the training run
model = YOLO('runs/train/weights/best.pt')

# Define path to the input video file
source = "video/100r_10.mp4"

# Run inference on the video in streaming mode (frame by frame)
results = model(source, stream=True, conf=0.25, iou=0.2, verbose=False)

# Initialize a counter for processed frames
n = 0

# Iterate over the inference results
for result in results:
    # Access bounding box outputs for the current frame
    boxes = result.boxes
    # Increment the frame counter
    n = n + 1

# Print the total number of frames processed
print(n)

#### YOLO Video Inference and Save Annotated Video

In [ ]:
import cv2
from ultralytics import YOLO

# Load the trained YOLO model
model = YOLO('runs/train/weights/best.pt')

# Open the input video file
video_path = "video/100r_10.mp4"
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f'Could not open video: {video_path}')

# Get video frame dimensions
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))

# Create a VideoWriter object to save the output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = int(cap.get(cv2.CAP_PROP_FPS))
out = cv2.VideoWriter(f'10_fps{fps}.mp4', fourcc, fps, (frame_width, frame_height))

# Loop through video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the frame
        results = model(frame, conf=0.25, iou=0.2, verbose=False)

        # Visualize the detections on the frame
        annotated_frame = results[0].plot()

        # Write the annotated frame to the output video
        out.write(annotated_frame)
    else:
        break

# Release the video capture and writer objects
cap.release()
out.release()


#### Real-Time YOLO Inference on Video with Display

In [ ]:
import cv2
from ultralytics import YOLO

# Load the trained YOLO model (YOLOv8 architecture)
model = YOLO('runs/train/weights/best.pt')

# Open the input video file
video_path = "video/100r_10.mp4"
cap = cv2.VideoCapture(video_path)

# Loop through video frames
while cap.isOpened():
    # Read one frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the current frame
        results = model(frame, conf=0.25, iou=0.2, verbose=False)

        # Visualize detections on the frame
        annotated_frame = results[0].plot()

        # Show the annotated frame in a window
        cv2.imshow("YOLO Inference", annotated_frame)

        # Exit loop when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # End of video reached
        break

# Release video capture and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()